# AR Reconciliation API - Endpoint Testing

## Setup
```
uv sync
uv run alembic upgrade head
uv run python main.py
```

Base URL: `http://localhost:8000`

In [2]:
import httpx
import time

BASE_URL = "http://localhost:8000"
client = httpx.Client(base_url=BASE_URL, timeout=30)
print("Client ready")

Client ready


## 1. Health Check
`GET /health` - Verify the server is running.

In [4]:
resp = client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'status': 'healthy', 'service': 'ar-reconciliation'}

## 2. Submit a Single Record
`POST /submit` - Submit one AR record for processing. Returns a workflow ID.

In [11]:
record = {
    "customer_id": "CUST-0001",
    "customer_name": "Customer 0001",
    "customer_balance": 3486.58,
    "invoice_total": 3892.38,
    "invoice_applied_amount": 241.57,
    "invoice_exchange_rate": 1.1371,
    "payment_total": 3316.88,
    "payment_applied_amount": 2375.22,
    "payment_exchange_rate": 0.8873,
    "credit_total": 910.95,
    "credit_applied_amount": 676.97,
    "credit_exchange_rate": 1.0237,
    "adjustment_total": 1310.49,
    "adjustment_applied_amount": 944.51,
    "adjustment_exchange_rate": 1.1211,
}

resp = client.post("/submit", json=record)
print(f"Status: {resp.status_code}")
submit_result = resp.json()
print(submit_result)
workflow_id = submit_result.get("workflow_id")
if workflow_id:
    print(f"Workflow ID: {workflow_id}")
else:
    print("No workflow_id in response: check server logs")


Status: 201
{'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484', 'status': 'COMPLETED', 'message': 'Duplicate submission - returning existing workflow'}
Workflow ID: d0e7376b-1be4-4ee7-bf63-91f55c2cc484


## 3. Submit Duplicate (Idempotency Check)
Submitting the same `customer_id` again should return the existing workflow, not create a new one.

In [12]:
# Submit the same record again - should be idempotent
resp = client.post("/submit", json=record)
print(f"Status: {resp.status_code}")
print("Expected: 'Duplicate submission' message")
resp.json()

Status: 201
Expected: 'Duplicate submission' message


{'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
 'status': 'COMPLETED',
 'message': 'Duplicate submission - returning existing workflow'}

## 4. Get Workflow Status
`GET /workflow/{id}` - Check the status and stage details of a workflow.

In [13]:
# Wait a moment for background processing to complete
time.sleep(2)

resp = client.get(f"/workflow/{workflow_id}")
print(f"Status: {resp.status_code}")
workflow_detail = resp.json()
print(f"Workflow status: {workflow_detail['workflow']['status']}")
print(f"Current stage: {workflow_detail['workflow']['current_stage']}")
print(f"Number of stages recorded: {len(workflow_detail['stages'])}")
workflow_detail

Status: 200
Workflow status: COMPLETED
Current stage: decision_routing
Number of stages recorded: 5


{'workflow': {'id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
  'invoice_id': 'CUST-0001',
  'customer_id': 'CUST-0001',
  'status': 'COMPLETED',
  'current_stage': 'decision_routing',
  'retry_count': 1,
  'created_at': '2026-05-28T13:38:57.941416',
  'updated_at': '2026-05-28T13:38:58.077777'},
 'stages': [{'id': 1,
   'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
   'stage_name': 'ingestion',
   'status': 'FAILED',
   'output_json': None,
   'error_message': 'Simulated random stage failure (POC demo)',
   'updated_at': '2026-05-28T13:38:57.980466'},
  {'id': 2,
   'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
   'stage_name': 'ingestion',
   'status': 'SUCCESS',
   'output_json': '{"status": "INGESTED", "record_id": "71efc100-ed90-488f-a901-9b1f95097a96"}',
   'error_message': None,
   'updated_at': '2026-05-28T13:38:58.014233'},
  {'id': 3,
   'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
   'stage_name': 'matching',
   'status': 'SUCCESS',
   'output_json'

## 5. Update a Record
`PUT /submit/{customer_id}` - Update an existing record and reprocess from scratch.

In [14]:
# Update the record with a different balance
updated_record = record.copy()
updated_record["customer_balance"] = 5000.00
updated_record["invoice_total"] = 5500.00

resp = client.put("/submit/CUST-0001", json=updated_record)
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'workflow_id': 'd0e7376b-1be4-4ee7-bf63-91f55c2cc484',
 'status': 'PENDING',
 'message': 'Record updated - workflow reprocessing from start'}

## 6. Bulk Upload CSV
`POST /bulk-upload` - Upload the CSV file to process multiple records at once.

In [15]:
# Upload the CSV file for bulk processing
with open("../data/erp_export.csv", "rb") as f:
    resp = client.post(
        "/bulk-upload", files={"file": ("erp_export.csv", f, "text/csv")}
    )

print(f"Status: {resp.status_code}")
bulk_result = resp.json()
print(f"Total records: {bulk_result['total_records']}")
print(f"Submitted: {bulk_result['submitted']}")
print(f"Duplicates: {bulk_result['duplicates']}")
print(f"Workflow IDs (first 5): {bulk_result['workflow_ids'][:5]}")
bulk_result

Status: 201
Total records: 1000
Submitted: 999
Duplicates: 1
Workflow IDs (first 5): ['d0e7376b-1be4-4ee7-bf63-91f55c2cc484', '2bbe9099-ac5a-44a8-828a-32c831aa8429', 'aac08e7f-c4ec-4f01-9c9f-58530c6dde63', '78ec058c-e7db-48c9-9d86-baf33d59843e', 'b8589cdd-c3f2-4480-84ec-8a0d74ca9191']


{'total_records': 1000,
 'submitted': 999,
 'duplicates': 1,
 'workflow_ids': ['d0e7376b-1be4-4ee7-bf63-91f55c2cc484',
  '2bbe9099-ac5a-44a8-828a-32c831aa8429',
  'aac08e7f-c4ec-4f01-9c9f-58530c6dde63',
  '78ec058c-e7db-48c9-9d86-baf33d59843e',
  'b8589cdd-c3f2-4480-84ec-8a0d74ca9191',
  '7b28c885-5e76-43d4-8825-7aa4a1a2d4b0',
  '9d3109c3-f70f-4b36-bcd7-15949ec9e124',
  '9fdda776-0012-4ade-8544-a542720dc646',
  '7229fa4f-767f-4752-984a-72f3ea151268',
  '6a1522de-2c9a-485f-a16a-c3f0d1a2e26b',
  '5d718f1f-0396-4535-9c8d-25a6ed393309',
  '86bc8c0f-4326-4cb5-9913-df4d673a1bd6',
  '9cd61bbf-2318-4232-8ff8-22ae2028c015',
  '03a1a4eb-1565-4cef-bfde-2f7a53c11acb',
  'c2db0fe9-7761-48eb-9eda-3ecedf4fabbb',
  '3baa4041-6322-4d9a-9ec7-e7c318f6188b',
  'b7b7d4c3-66bf-47da-a4c5-139cb0885252',
  '7abb39f0-d742-427c-a3aa-27e5cf3ae138',
  'b76fc0bc-4e96-4927-9b77-bc66b8b70c13',
  '64c14378-c1ec-4c9b-aca7-181ff5cdf7b0',
  'f359ecc0-0098-4bc6-95bd-a730d9075d65',
  '04ada03a-45ba-42bf-995d-1d498aa269a0',

## 7. List All Workflows
`GET /workflows` - List workflows with optional status filter and pagination.

In [16]:
# Wait for background processing
time.sleep(3)

# List all workflows
resp = client.get("/workflows")
print(f"Status: {resp.status_code}")
workflows = resp.json()
print(f"Total workflows returned: {len(workflows)}")
# Show first 3
for w in workflows[:3]:
    print(
        f"  {w['id']} | {w['customer_id']} | {w['status']} | stage: {w['current_stage']}"
    )

Status: 200
Total workflows returned: 50
  4eec10bb-c22a-4dc0-9456-6a8315bc221a | CUST-1000 | PENDING | stage: None
  d48d869c-084d-40f9-9c45-9bc1ae5566f4 | CUST-0999 | PENDING | stage: None
  c40c84d4-e7e4-4594-ae03-672a3681b7a3 | CUST-0998 | PENDING | stage: None


In [17]:
# Filter workflows by status
resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 5})
print(f"Completed workflows: {resp.status_code}")
print(f"Count: {len(resp.json())}")

resp = client.get("/workflows", params={"status": "FAILED", "limit": 5})
print(f"\nFailed workflows: {resp.status_code}")
failed_workflows = resp.json()
print(f"Count: {len(failed_workflows)}")
for w in failed_workflows[:3]:
    print(f"  {w['id']} | {w['customer_id']} | retry_count: {w['retry_count']}")

Completed workflows: 200
Count: 5

Failed workflows: 200
Count: 5
  e62f0b38-d0b3-4acd-9001-6fa361425465 | CUST-0281 | retry_count: 3
  c8e4074f-6396-4fb8-9e1d-67bfb55f9ab7 | CUST-0273 | retry_count: 3
  2f6f5c57-a66d-45b0-a771-3c9c90fd136d | CUST-0251 | retry_count: 3


## 8. Resume a Failed Workflow
`POST /resume/{id}` - Resume a workflow that failed (retries from the last successful stage).

In [18]:
# Find a failed workflow to resume
resp = client.get("/workflows", params={"status": "FAILED", "limit": 1})
failed = resp.json()

if failed:
    failed_id = failed[0]["id"]
    print(f"Resuming workflow: {failed_id}")
    resp = client.post(f"/resume/{failed_id}")
    print(f"Status: {resp.status_code}")
    print(resp.json())

    # Check status after a moment
    time.sleep(2)
    resp = client.get(f"/workflow/{failed_id}")
    print(f"\nAfter resume - Status: {resp.json()['workflow']['status']}")
else:
    print("No failed workflows to resume (all succeeded!)")
    print(
        "Note: The system has a 20% random failure rate, so run bulk-upload again if needed"
    )

Resuming workflow: 7c8a046c-283a-4276-8de1-9cce17590eb7
Status: 200
{'workflow_id': '7c8a046c-283a-4276-8de1-9cce17590eb7', 'status': 'COMPLETED', 'message': 'Workflow already completed all stages'}

After resume - Status: FAILED


## 9. Dashboard Stats
`GET /stats` - Get aggregate statistics about all workflows.

In [20]:
resp = client.get("/stats")
print(f"Status: {resp.status_code}")
stats = resp.json()
print(f"Total workflows: {stats['total_workflows']}")
print(f"Completed: {stats['completed']}")
print(f"Failed: {stats['failed']}")
print(f"Pending: {stats['pending']}")
print(f"Running: {stats['running']}")
print(f"Stale: {stats['stale']}")
print(f"Routing decisions: {stats['decisions']}")

Status: 200
Total workflows: 1000
Completed: 966
Failed: 34
Pending: 0
Running: 0
Stale: 0
Routing decisions: {'MANUAL_REVIEW': 577, 'FINANCE_REVIEW': 351, 'AUTO_APPROVED': 38}


## 10. Enhanced Workflow List
`GET /workflows/enhanced` - List workflows with staleness flag and last error message.

In [ ]:
# Enhanced list with error details
resp = client.get("/workflows/enhanced", params={"limit": 5})
print(f"Status: {resp.status_code}")
enhanced = resp.json()
for w in enhanced[:5]:
    stale_flag = " [STALE]" if w.get("is_stale") else ""
    error = f" | error: {w['last_error']}" if w.get("last_error") else ""
    print(f"  {w['customer_id']} | {w['status']}{stale_flag}{error}")

## 11. Export Results as CSV
`GET /export` - Download completed workflow results as a CSV file.

In [21]:
# Export completed results as CSV
resp = client.get("/export")
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('content-type')}")
print("\nCSV Preview (first 500 chars):")
print(resp.text[:500])

Status: 200
Content-Type: text/csv; charset=utf-8

CSV Preview (first 500 chars):
workflow_id,customer_id,status,match_result,is_valid,decision,high_value,retry_count,completed_at
d0e7376b-1be4-4ee7-bf63-91f55c2cc484,CUST-0001,COMPLETED,PARTIAL,True,MANUAL_REVIEW,False,2,2026-05-28 13:40:24.835313
2bbe9099-ac5a-44a8-828a-32c831aa8429,CUST-0002,COMPLETED,OVERPAID,True,FINANCE_REVIEW,False,1,2026-05-28 13:41:34.894885
aac08e7f-c4ec-4f01-9c9f-58530c6dde63,CUST-0003,COMPLETED,PARTIAL,True,MANUAL_REVIEW,False,0,2026-05-28 13:41:35.148160
78ec058c-e7db-48c9-9d86-baf33d59843e,CU


## 12. Error Handling - Invalid Requests
Test that the API returns proper error responses for bad input.

In [22]:
# Test 404 - workflow not found
resp = client.get("/workflow/nonexistent-id-12345")
print(f"GET /workflow/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 404 - resume non-existent workflow
resp = client.post("/resume/nonexistent-id-12345")
print(f"\nPOST /resume/bad-id -> {resp.status_code} (expected 404)")
print(f"  Response: {resp.json()}")

# Test 400 - upload non-CSV file
resp = client.post(
    "/bulk-upload", files={"file": ("test.txt", b"not a csv", "text/plain")}
)
print(f"\nPOST /bulk-upload with .txt -> {resp.status_code} (expected 400)")
print(f"  Response: {resp.json()}")

# Test 422 - missing required field
resp = client.post("/submit", json={})
print(f"\nPOST /submit with empty body -> {resp.status_code} (expected 422)")
print(f"  Response: {resp.json()['detail'][0]['msg']}")

GET /workflow/bad-id -> 404 (expected 404)
  Response: {'detail': 'Workflow not found'}

POST /resume/bad-id -> 404 (expected 404)
  Response: {'detail': 'Workflow not found'}

POST /bulk-upload with .txt -> 400 (expected 400)
  Response: {'detail': 'Only CSV files are supported'}

POST /submit with empty body -> 422 (expected 422)
  Response: Field required


## 13. Resume All Failed Workflows
Loop through all failed workflows and attempt to resume them.

In [26]:
# Resume all failed workflows
resp = client.get("/workflows", params={"status": "FAILED", "limit": 200})
failed = resp.json()
print(f"Found {len(failed)} failed workflows. Resuming...")

for w in failed:
    r = client.post(f"/resume/{w['id']}")
    print(f"  {w['customer_id']}: {r.json()['message']}")

# Wait and check final stats
time.sleep(5)
resp = client.get("/stats")
stats = resp.json()
print("\n--- Final Stats ---")
print(f"Completed: {stats['completed']}/{stats['total_workflows']}")
print(f"Still failed: {stats['failed']}")
print(f"Decisions: {stats['decisions']}")

Found 9 failed workflows. Resuming...
  CUST-0995: Workflow already completed all stages
  CUST-0972: Workflow already completed all stages
  CUST-0864: Workflow already completed all stages
  CUST-0741: Workflow already completed all stages
  CUST-0508: Workflow already completed all stages
  CUST-0369: Workflow already completed all stages
  CUST-0350: Workflow already completed all stages
  CUST-0246: Workflow already completed all stages
  CUST-0199: Workflow already completed all stages

--- Final Stats ---
Completed: 992/1001
Still failed: 9
Decisions: {'MANUAL_REVIEW': 584, 'FINANCE_REVIEW': 356, 'AUTO_APPROVED': 39, 'COLLECTION_QUEUE': 6, 'REJECTED': 7}


---

## 14. Concurrent Duplicate Submission
Fire multiple requests for the same customer_id at once - only one workflow should be created.


In [27]:
import concurrent.futures

# Use a unique customer ID for this test
test_customer_id = "RACE-COND-TEST-001"
race_record = {
    "customer_id": test_customer_id,
    "customer_name": "Race Condition Test",
    "invoice_total": 1000.00,
    "payment_total": 1000.00,
}

def submit_record(record):
    """Submit in a separate thread to simulate concurrency."""
    c = httpx.Client(base_url=BASE_URL, timeout=30)
    return c.post("/submit", json=record).json()

# Fire 5 concurrent requests for the same customer
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(submit_record, race_record) for _ in range(5)]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

# All should return the same workflow_id (no duplicates created)
workflow_ids = set(r["workflow_id"] for r in results)
statuses = [r["status"] for r in results]

print("Concurrent submissions: 5")
print(f"Unique workflow IDs returned: {len(workflow_ids)} (expected: 1)")
print(f"Workflow ID: {workflow_ids.pop()}")
print(f"Statuses: {statuses}")
assert len(workflow_ids) == 0, "ERROR: Multiple workflows created for same customer!"
print("\n[ok] Race condition handled correctly - no duplicate workflows")


Concurrent submissions: 5
Unique workflow IDs returned: 1 (expected: 1)
Workflow ID: ba23692d-26c8-43de-ad53-b696ea943ca9
Statuses: ['COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPLETED', 'COMPLETED']

[ok] Race condition handled correctly - no duplicate workflows


## 15. ACID / Atomicity Check
Submit same record twice, confirm idempotency key dedup works and stages are persisted atomically.


In [28]:
# Submit the same record data twice via the API - second should be idempotent
acid_record = {
    "customer_id": "ACID-TEST-001",
    "customer_name": "ACID Test Customer",
    "invoice_total": 2500.00,
    "payment_total": 2500.00,
}

# First submission
resp1 = client.post("/submit", json=acid_record)
result1 = resp1.json()
print(f"1st submit: status={result1['status']}, workflow_id={result1['workflow_id']}")

time.sleep(2)

# Second submission (same data) - should return DUPLICATE via idempotency
resp2 = client.post("/submit", json=acid_record)
result2 = resp2.json()
print(f"2nd submit: status={result2['status']}, workflow_id={result2['workflow_id']}")

# Verify same workflow is returned
assert result1["workflow_id"] == result2["workflow_id"], "ERROR: Different workflow IDs!"
print("\n[ok] Idempotency verified - same workflow returned for duplicate data")

# Verify workflow completed with all stages atomically persisted
resp = client.get(f"/workflow/{result1['workflow_id']}")
detail = resp.json()
print(f"\nWorkflow status: {detail['workflow']['status']}")
print(f"Stages recorded: {len(detail['stages'])}")
for stage in detail['stages']:
    if stage['status'] == 'SUCCESS':
        print(f"  [+] {stage['stage_name']}: {stage['status']}")
    else:
        print(f"  X {stage['stage_name']}: {stage['status']} - {stage.get('error_message', '')}")


1st submit: status=PENDING, workflow_id=6d72e103-56c2-4c17-af17-b0f503eeebdd
2nd submit: status=COMPLETED, workflow_id=6d72e103-56c2-4c17-af17-b0f503eeebdd

[ok] Idempotency verified - same workflow returned for duplicate data

Workflow status: COMPLETED
Stages recorded: 5
  [+] ingestion: SUCCESS
  [+] matching: SUCCESS
  X validation: FAILED - Simulated random stage failure (POC demo)
  [+] validation: SUCCESS
  [+] decision_routing: SUCCESS


## 16. Retry Exhaustion
Confirm workflows end up in FAILED (not stuck in RUNNING) when retries are exhausted.


In [29]:
# Submit multiple records to exercise the retry mechanism
retry_records = [
    {"customer_id": f"RETRY-TEST-{i:03d}", "customer_name": f"Retry Test {i}",
     "invoice_total": 1000.0 + i, "payment_total": 900.0 + i}
    for i in range(10)
]

submitted_ids = []
for rec in retry_records:
    resp = client.post("/submit", json=rec)
    result = resp.json()
    submitted_ids.append(result["workflow_id"])

print(f"Submitted {len(submitted_ids)} records for retry testing")
print("Waiting for background processing (with 20% failure rate)...")
time.sleep(5)

# Check that NO workflow is stuck in RUNNING state
resp = client.get("/workflows", params={"status": "RUNNING", "limit": 200})
running = resp.json()
print(f"\nWorkflows stuck in RUNNING: {len(running)} (expected: 0)")

# Check retry counts on failed workflows
resp = client.get("/workflows", params={"status": "FAILED", "limit": 200})
failed = resp.json()
print(f"Workflows that FAILED after retries: {len(failed)}")
for w in failed[:5]:
    print(f"  {w['customer_id']} | retry_count={w['retry_count']} | stage={w['current_stage']}")

resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 200})
completed = resp.json()
print(f"Workflows COMPLETED: {len(completed)}")

if len(running) == 0:
    print("\n[ok] No workflows stuck in RUNNING - failure handling works correctly")
else:
    print("\nX WARNING: Some workflows stuck in RUNNING state!")


Submitted 10 records for retry testing
Waiting for background processing (with 20% failure rate)...

Workflows stuck in RUNNING: 0 (expected: 0)
Workflows that FAILED after retries: 10
  RETRY-TEST-002 | retry_count=3 | stage=validation
  CUST-0995 | retry_count=6 | stage=decision_routing
  CUST-0972 | retry_count=3 | stage=decision_routing
  CUST-0864 | retry_count=4 | stage=decision_routing
  CUST-0741 | retry_count=4 | stage=decision_routing
Workflows COMPLETED: 200

[ok] No workflows stuck in RUNNING - failure handling works correctly


## 17. Resume from Checkpoint
Resume a failed workflow and confirm it picks up from the last successful stage.


In [30]:
# Find a failed workflow that has some successful stages
resp = client.get("/workflows", params={"status": "FAILED", "limit": 10})
failed = resp.json()

resumed_wf = None
for w in failed:
    detail_resp = client.get(f"/workflow/{w['id']}")
    detail = detail_resp.json()
    successful_stages = [s for s in detail["stages"] if s["status"] == "SUCCESS"]
    failed_stages = [s for s in detail["stages"] if s["status"] == "FAILED"]
    
    if successful_stages:
        resumed_wf = w
        print(f"Found workflow with partial progress: {w['id']}")
        print(f"  Customer: {w['customer_id']}")
        print(f"  Successful stages: {[s['stage_name'] for s in successful_stages]}")
        print(f"  Failed at: {w['current_stage']} (retry_count={w['retry_count']})")
        break

if resumed_wf:
    # Resume it
    resp = client.post(f"/resume/{resumed_wf['id']}")
    print(f"\nResume response: {resp.json()['message']}")
    time.sleep(3)
    
    # Check if it progressed further
    resp = client.get(f"/workflow/{resumed_wf['id']}")
    detail = resp.json()
    print(f"After resume - status: {detail['workflow']['status']}")
    print("Stage history:")
    for s in detail["stages"]:
        marker = "[+]" if s["status"] == "SUCCESS" else "X"
        print(f"  {marker} {s['stage_name']}: {s['status']}")
    print("\n[ok] Resume from checkpoint works - state was reconstructed correctly")
else:
    print("No failed workflows with partial progress found (all may have completed)")
    print("This is expected when failure rate is low or all retries succeeded")


Found workflow with partial progress: 1690967f-23b0-4516-8fb0-3324b803ba86
  Customer: RETRY-TEST-002
  Successful stages: ['ingestion', 'matching']
  Failed at: validation (retry_count=3)

Resume response: Workflow resuming from after stage: validation
After resume - status: COMPLETED
Stage history:
  [+] ingestion: SUCCESS
  [+] matching: SUCCESS
  X validation: FAILED
  X validation: FAILED
  X validation: FAILED
  [+] decision_routing: SUCCESS

[ok] Resume from checkpoint works - state was reconstructed correctly


## 18. DB Consistency Audit
Check that COMPLETED workflows have all 4 stages, FAILED ones have recorded errors, none stuck in RUNNING.


In [31]:
# Audit all workflows for DB consistency
resp = client.get("/workflows", params={"limit": 200})
all_workflows = resp.json()

inconsistent = []
for w in all_workflows:
    detail_resp = client.get(f"/workflow/{w['id']}")
    detail = detail_resp.json()
    stages = detail["stages"]
    successful = [s for s in stages if s["status"] == "SUCCESS"]
    
    if w["status"] == "COMPLETED":
        # Completed workflows must have all 4 stages successful
        if len(successful) < 4:
            inconsistent.append(f"COMPLETED but only {len(successful)} successful stages: {w['id']}")
    elif w["status"] == "FAILED":
        # Failed workflows must have at least one FAILED stage recorded
        failed_stages = [s for s in stages if s["status"] == "FAILED"]
        if not failed_stages:
            inconsistent.append(f"FAILED but no failed stages recorded: {w['id']}")
    elif w["status"] == "RUNNING":
        # No workflow should be stuck in RUNNING (pipeline guard catches this)
        inconsistent.append(f"Stuck in RUNNING: {w['id']}")

print(f"Total workflows audited: {len(all_workflows)}")
print(f"Inconsistencies found: {len(inconsistent)}")

if inconsistent:
    for issue in inconsistent:
        print(f"  X {issue}")
else:
    print("\n[ok] All workflows are in a consistent state - ACID properties maintained")
    
# Summary
statuses = {}
for w in all_workflows:
    statuses[w["status"]] = statuses.get(w["status"], 0) + 1
print(f"\nStatus distribution: {statuses}")


Total workflows audited: 200
Inconsistencies found: 4
  X COMPLETED but only 3 successful stages: 1690967f-23b0-4516-8fb0-3324b803ba86
  X COMPLETED but only 3 successful stages: 2c1d7408-7563-4b79-ba1a-0947dc039b51
  X COMPLETED but only 3 successful stages: 15c2b155-bfd4-4998-854a-5af27100614e
  X COMPLETED but only 3 successful stages: 34ff9587-f791-4dbf-8d5c-b15718227e01

Status distribution: {'COMPLETED': 197, 'FAILED': 3}


## 19. State Machine Checks
- Can't resume a COMPLETED workflow
- Non-existent workflow returns 404
- Only valid statuses exist in DB


In [32]:
# Test: Cannot resume a COMPLETED workflow
resp = client.get("/workflows", params={"status": "COMPLETED", "limit": 1})
completed = resp.json()

if completed:
    wf_id = completed[0]["id"]
    resp = client.post(f"/resume/{wf_id}")
    result = resp.json()
    print(f"Resume completed workflow: {result['message']}")
    assert "already completed" in result["message"].lower(), "Should indicate already completed"
    print("[ok] Completed workflow correctly rejects resume")
else:
    print("No completed workflows to test (run bulk upload first)")

# Test: Resume non-existent workflow returns 404
resp = client.post("/resume/does-not-exist-xyz")
print(f"\nResume non-existent workflow: {resp.status_code} (expected 404)")
assert resp.status_code == 404
print("[ok] Non-existent workflow correctly returns 404")

# Test: Check that workflow statuses are only valid values
resp = client.get("/workflows", params={"limit": 200})
all_wf = resp.json()
valid_statuses = {"PENDING", "RUNNING", "COMPLETED", "FAILED"}
invalid = [w for w in all_wf if w["status"] not in valid_statuses]
print(f"\nWorkflows with invalid status: {len(invalid)} (expected: 0)")
assert len(invalid) == 0, f"Found invalid statuses: {[w['status'] for w in invalid]}"
print("[ok] All workflow statuses are valid state machine values")


Resume completed workflow: Workflow already completed
[ok] Completed workflow correctly rejects resume

Resume non-existent workflow: 404 (expected 404)
[ok] Non-existent workflow correctly returns 404

Workflows with invalid status: 0 (expected: 0)
[ok] All workflow statuses are valid state machine values
